<a href="https://colab.research.google.com/github/Sena-Lee-97/Code/blob/main/skin_disease_classification_using_custom_cn_d5e4b8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
kmader_skin_cancer_mnist_ham10000_path = kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')

print('Data source import complete.')


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
import os
from PIL import Image
from sklearn.preprocessing import MinMaxScaler
import cv2
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input, Concatenate, Dropout, Flatten, MaxPooling2D, Conv2D
from tensorflow.keras.models import Model

from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import random
from sklearn.utils import class_weight
import random
from tensorflow.keras.optimizers import Adam, SGD, RMSprop, Nadam, Adagrad, Adadelta
from tensorflow.keras import layers
from collections import Counter
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix
import seaborn as sns
from sklearn.metrics import classification_report

In [ ]:
images = pd.read_csv("/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv")

In [ ]:
images['localization'].value_counts()

In [ ]:
images.info()


In [ ]:
images["dx"].value_counts()      #shows the frequency of number of classes


In [ ]:
#frequency bar graph of the varying classes

counts = images['dx'].value_counts()

plt.figure(figsize=(10,6))
plt.bar(counts.index,counts.values, color='skyblue', edgecolor = 'black')

plt.xlabel('Classes' , fontsize =12)
plt.ylabel('Frequencis', fontsize = 12)
plt.title('Frequency of skin cancer classes ')

plt.show()

In [ ]:
#here we have trimmed down the frequency of nv class to 1000
nv_images = images[images['dx'] == 'nv']

# Randomly sample 1000 rows from 'nv'
nv_sampled = nv_images.sample(n=1000, random_state=42)

# Keep all other classes unchanged
other_images = images[(images['dx'] != 'nv')&(images['dx']!='df')&(images['dx']!='vasc')]

# Combine sampled 'nv' rows with other classes
images_nv_trimmed = pd.concat([nv_sampled, other_images], ignore_index=True)

# Shuffle the combined DataFrame (optional but recommended)
images_nv_trimmed = images_nv_trimmed.sample(frac=1, random_state=42).reset_index(drop=True)


In [ ]:
# bkl, mel만 선택
target_classes = ["bkl", "mel"]

images_binary = images_nv_trimmed[
    images_nv_trimmed["dx"].isin(target_classes)
].reset_index(drop=True)

images_binary["dx"].value_counts()

# =====================================================
# 🔥 [추가] 이진 분류용 라벨 매핑
# =====================================================
label_map = {"bkl": 0, "mel": 1}
images_binary["dx_encoded"] = images_binary["dx"].map(label_map)


In [ ]:
#Lets see the improvised frequency bar graph

counts = images_nv_trimmed['dx'].value_counts()

plt.figure(figsize=(10,6))
plt.bar(counts.index,counts.values, color='skyblue', edgecolor = 'black')

plt.xlabel('Classes' , fontsize =12)
plt.ylabel('Frequencis', fontsize = 12)
plt.title('Frequency of skin cancer classes ')






In [ ]:
#as age consists some missing values for that we will fill the data with median in that place
median_age = images_nv_trimmed['age'].median()
images_nv_trimmed['age'] = images_nv_trimmed['age'].fillna(median_age)
images_nv_trimmed.info()



In [ ]:
images_nv_trimmed['sex'].value_counts()

In [ ]:
#in the sex column replacing the unknown value with 'female' in order to balance
images_nv_trimmed['sex'] = images_nv_trimmed['sex'].replace('unknown', 'female')

In [ ]:
images_nv_trimmed['sex'].value_counts()

In [ ]:
encoder = LabelEncoder()
encoded_df = pd.DataFrame(columns=['image_id','dx_encoded', 'sex_encoded', 'loc_encoded','dx_type_encoded'])

encoded_df['image_id'] = images_binary['image_id']
encoded_df['dx_encoded'] = images_binary['dx_encoded']

encoded_df['sex_encoded'] = encoder.fit_transform(images_nv_trimmed['sex'])
encoded_df['loc_encoded'] = encoder.fit_transform(images_nv_trimmed['localization'])
encoded_df['dx_type_encoded']=encoder.fit_transform(images_nv_trimmed['dx_type'])
encoded_df['age'] = images_nv_trimmed['age'].copy()


In [ ]:
encoded_df.columns

In [ ]:
X = encoded_df.drop(['dx_encoded', 'image_id'],axis = 1)
Y = encoded_df['dx_encoded']

X.info()


In [ ]:
#Now we analyze the mutual information in order to select the feature having highest scores

mi = mutual_info_classif(X,Y, discrete_features='auto')
mi_scores = pd.Series(mi, index = X.columns).sort_values(ascending = False)

mi_scores


In [ ]:
#As seen from the Mutual information score dx_type, localization and age are the most important features
#We now drop the sex column from our dataset

X = X.drop('sex_encoded', axis =1)

In [ ]:
X.head(10)

In [ ]:
Y = images_nv_trimmed['dx'].copy()
Y.head(10)

In [ ]:
X['image_id'] = images_nv_trimmed['image_id']

In [ ]:
X.columns

In [ ]:
cat_cols = ['loc_encoded', 'dx_type_encoded']
for col in cat_cols:
    X[col].astype('category')

In [ ]:
#One-hot encoded the categorical columns
X = pd.get_dummies(X , columns=cat_cols)

In [ ]:

scaler = MinMaxScaler()
X['age_scaled'] = scaler.fit_transform(encoded_df[['age']])
X=X.drop('age', axis =1)


In [ ]:

X['dx_encoded'] = encoded_df['dx_encoded'].copy()

In [ ]:
def get_image_path(img_id):
    if os.path.exists(f"/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1/{img_id}.jpg"):
        return f"/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1/{img_id}.jpg"
    else:
        return f"/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2/{img_id}.jpg"

image_paths = X['image_id'].map(get_image_path)
image_paths

In [ ]:
X['images_paths'] = image_paths


In [ ]:
X.columns

In [ ]:
#Keeping constants here
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
NUM_CLASSES = 2

In [ ]:
#Train Test Spilt
train_df, temp_df = train_test_split(
    X,
    test_size=0.2,
    random_state=42,
    stratify=X["dx_encoded"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,  # half of 20% → 10% val, 10% test
    random_state=42,
    stratify=temp_df["dx_encoded"]
)


In [ ]:
class_counts = train_df["dx_encoded"].value_counts().sort_index()
num_classes = len(class_counts)
total_samples = len(train_df)

alpha = 0.5          # √-scaled
MAX_WEIGHT = 3.0     # safety clamp

class_weights = {
    cls: min((total_samples / (num_classes * count)) ** alpha, MAX_WEIGHT)
    for cls, count in class_counts.items()
}

print(class_weights)

In [ ]:
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.keras.applications.resnet50.preprocess_input(img)

    label = tf.one_hot(label, NUM_CLASSES)
    return img, label


data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomZoom(0.2, 0.2),
        tf.keras.layers.RandomTranslation(0.2, 0.2),
        tf.keras.layers.RandomBrightness(0.0),
    ],
    name="augmentation"
)

def load_image_aug(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)

    img = data_augmentation(img)

    img = tf.keras.applications.resnet50.preprocess_input(img)
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label


def build_dataset(df, augment=False, shuffle=True):
    paths = df["images_paths"].values
    labels = df["dx_encoded"].values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(df))

    if augment:
        ds = ds.map(load_image_aug, num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


In [ ]:
train_ds = build_dataset(train_df, augment=False)
train_aug_ds = build_dataset(train_df, augment=True)
val_ds = build_dataset(val_df, augment=False, shuffle=False)
test_ds = build_dataset(test_df, augment=False, shuffle=False)

In [ ]:
def resnet50_feature_extractor(input_shape=(224, 224, 3)):
    base = tf.keras.applications.ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=input_shape
    )
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    return tf.keras.Model(base.input, x, name="mod1_f")

def lightweight_feature_extractor(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(16, 3, activation="relu", kernel_initializer="he_normal")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32, 3, activation="relu", kernel_initializer="he_normal")(x)
    x = layers.GlobalAveragePooling2D()(x)
    return tf.keras.Model(inputs, x, name="mod2_f")

def build_final_fusion_model(num_classes=7):
    inputs = layers.Input(shape=(224, 224, 3), name="input_layer")

    mod1 = resnet50_feature_extractor()
    mod2 = lightweight_feature_extractor()

    f1 = mod1(inputs)
    f2 = mod2(inputs)

    # Conv2D + Activation (as in diagram)
    f1 = layers.Reshape((1, 1, -1))(f1)
    f1 = layers.Conv2D(64, 1, activation="relu")(f1)

    f2 = layers.Reshape((1, 1, -1))(f2)
    f2 = layers.Conv2D(64, 1, activation="relu")(f2)

    # Concatenate
    x = layers.Concatenate()([f1, f2])

    # Flatten
    x = layers.Flatten()(x)

    # Dense → Activation
    x = layers.Dense(256, activation="relu", kernel_initializer="he_normal")(x)

    # Dropout
    x = layers.Dropout(0.5)(x)

    # Output
    outputs = layers.Dense(num_classes, activation="sigmoid", name="dense_3")(x)

    return tf.keras.Model(inputs, outputs, name="Final_Fusion_Model")


In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    mode="max",
    restore_best_weights=True,
    min_delta=0.001,
    verbose=1
)

In [ ]:
model = build_final_fusion_model()

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss = "binary_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weights,

    callbacks=[early_stopping]

)


In [ ]:
resnet = model.get_layer("mod1_f")

for layer in resnet.layers[-30:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train_aug_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights
)

In [ ]:
for layer in resnet.layers[-80:-30]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-6),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history=model.fit(
    train_aug_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
epochs = range(1, len(acc) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, acc, label="Training Accuracy")
plt.plot(epochs, val_acc, label="Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
y_true = []
y_pred = []

for x, y in test_ds:
    preds = model.predict(x, verbose=0)
    y_true.extend(tf.argmax(y, axis=1).numpy())
    y_pred.extend(tf.argmax(preds, axis=1).numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=range(NUM_CLASSES),
    yticklabels=range(NUM_CLASSES)
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix – Final Model (Test Set)")
plt.show()

print(classification_report(y_true, y_pred, digits=4))

In [ ]:
model.save("/kaggle/working/ham10000_final_model.keras")
